# Timing 分组校准 Demo · QXtrlTimingCali v0.4（六步流程）

生成芯片配置 → 造模拟环境（真值 L）→ 生成待测文件 → **按物理顺序做六步** →
与真值对账。

| 步 | 实验 | 数据怎么处置 | 说明 |
| --- | --- | --- | --- |
| **1** | 读取线校准 `RL.ri ↔ RL.ro` | **测完立刻求解并写回** | 读取窗口没对准，后面什么都读不出 |
| **2** | 读取线整体右移 `headroom` | **立刻写回** | 给 qubit/coupler 留出右移空间（典型 0–1000 ns） |
| **3** | 单比特 XY-Z 校准 `Q.xy ↔ Q.z` | 只记录，先不写 | 读取线可用后才能逐比特对准 |
| **4** | 比特间 Z-Z / 耦合器 Z 校准 | 只记录，先不写 | 可调比特用 `Q.z`，不可调用 `Q.xy` |
| **5** | 统一求解 qubit + coupler | **整批一次写回** | 步 3、步 4（组 `qubit` / `coupler`）同基线，合并做一次全局最小二乘 |
| **6** | 读取线收回 `headroom`、拉回对齐 | **立刻写回** | 以实测 skew 收尾，把步 2 主动右移的余量往左收回 |

为什么这样排：

- 读取脉冲的发射(`ro`)与采样窗口(`ri`)没对准，后面**什么都读不出来**，X_π 更无从谈起，
  所以步骤 1 不能等：测完就地落盘；
- 步骤 2 的 `headroom` 是经验性预留：真实机台不知道 qubit/coupler 最终位置，先给读取线
  一个额外右移，避免后续 qubit 右移时撞到读取窗口。数值可按实测反馈调整；
- 步骤 3、步骤 4 之间**没有任何写回**，两组 `skew` 描述同一时刻的物理状态，
  因此步骤 5 可以把它们合并求解；
- 步骤 6 发生在 qubit/coupler 全局对齐之后：以 readout ↔ 首腔 `qubit.xy` 的实测 skew
  做收尾求解，把步骤 2 主动右移的 `headroom` **往左收回**（受授予台账约束，
  `d` 不会低于加 headroom 之前的校准基线），读取线最终与 qubit 门时序一致。

- 库本身**零第三方依赖**，本 notebook 只额外用到 Jupyter（画图用自绘 SVG，不用 matplotlib）；
- 中间产物落在仓库的 `runs/demo/`（已在 `.gitignore` 内），从 `examples/` 或仓库根启动均可；
- 数据主线：某条信号线的时序位置 `q = L + d`。`L` 是设备隐藏真值（truth 文件，校准流程不读），
  `d` 是累计右移（待测文件，校准流程只增不减地回写它）。


In [1]:
import json
import sys
from collections import Counter
from html import escape
from pathlib import Path

from IPython.display import HTML, Markdown, display

HERE = Path.cwd()

def _find_repo(start):
    """从当前目录向上找到仓库根：无论 notebook 从 examples/ 还是仓库根启动都能导入包。"""
    for cand in [start, *start.parents]:
        if (cand / "qxtrl_timing_cali").is_dir():
            return cand
    raise RuntimeError("未找到 qxtrl_timing_cali 包目录，请在仓库内启动本 notebook")

REPO = _find_repo(HERE)
sys.path.insert(0, str(REPO))
WORK = REPO / "runs" / "demo"        # 与 CLI 同用 runs/ 目录（已 gitignore）
WORK.mkdir(parents=True, exist_ok=True)

from qxtrl_timing_cali import (__version__, generate_chip, generate_wiring, line_names,
                               render_chip_html, save_chip, save_wiring,
                               validate_chip, validate_wiring,
                               CalibrationSession, ReviewCenter, Thresholds, DirectExecutor,
                               SimChip, SimExecutor, position_residual,
                               StagedCalibrationPipeline, plan_stages)


def check(tag, errs, warns):
    """校验器返回 (errors, warnings)：errors 非空即中止，warnings 只打印。"""
    print(f"[{tag}] errors={len(errs)}  warnings={len(warns)}")
    for line in errs + warns:
        print("      ", line)
    assert not errs, f"[{tag}] 校验未通过"


def show(doc, height=640):
    """把 render_chip_html 的自包含 HTML 内嵌进单元格（srcdoc，不依赖服务器路径）。"""
    return HTML(f'<iframe srcdoc="{escape(doc, quote=True)}" title="chip-viz" '
                f'style="width:100%;height:{height}px;border:1px solid #d7dee6;'
                f'border-radius:8px"></iframe>')


def shift_strip(title, rows):
    """自绘位置条：rows = [(线名, L, d)]。灰点 = 初始 L，绿条 = 右移 d，蓝点 = 校准后 q = L + d。"""
    lm, w, rh = 112, 900, 15
    h = 56 + rh * len(rows)
    vals = [v for _, l, d in rows for v in (l, l + d)]
    lo, hi = min(vals), max(vals)

    def X(v):
        return lm + (v - lo) * (w - lm - 34) / max(hi - lo, 1e-9)

    p = [f'<svg viewBox="0 0 {w} {h}" xmlns="http://www.w3.org/2000/svg" '
         f'style="background:#fff;border:1px solid #d7dee6;border-radius:8px;'
         f'font-family:system-ui,sans-serif">',
         f'<text x="14" y="21" font-size="13" font-weight="600" fill="#20303f">{escape(title)}</text>',
         '<text x="14" y="35" font-size="10.5" fill="#5c6b7a">● 初始 L　▬ 右移 d　● 校准后 q = L + d'
         f'</text>',
         f'<line x1="{lm}" y1="42" x2="{lm}" y2="{h - 16}" stroke="#c8d2dc"/>',
         f'<line x1="{w - 34}" y1="42" x2="{w - 34}" y2="{h - 16}" stroke="#c8d2dc"/>']
    for i, (name, l, d) in enumerate(rows):
        y = 48 + i * rh
        p += [f'<text x="{lm - 8}" y="{y + 3.5}" font-size="9.5" text-anchor="end" '
              f'fill="#5c6b7a">{escape(name)}</text>',
              f'<line x1="{X(l):.1f}" y1="{y}" x2="{X(l + d):.1f}" y2="{y}" '
              f'stroke="#1d7a3c" stroke-width="3"/>',
              f'<circle cx="{X(l):.1f}" cy="{y}" r="2.8" fill="#8a97a4"/>',
              f'<circle cx="{X(l + d):.1f}" cy="{y}" r="3.2" fill="#2c6cb0"/>']
    p += [f'<text x="{lm}" y="{h - 4}" font-size="10" fill="#5c6b7a">{lo:.0f} ns</text>',
          f'<text x="{w - 34}" y="{h - 4}" font-size="10" text-anchor="end" '
          f'fill="#5c6b7a">{hi:.0f} ns</text>', "</svg>"]
    return HTML("".join(p))


def md_table(headers, rows):
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    out += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    return "\n".join(out)


def disk_delays(path):
    """回读磁盘上的待测文件（不是内存里的会话）。"""
    return {e["line"]: float(e["delay"])
            for e in json.loads(Path(path).read_text(encoding="utf-8"))["lines"]}


def stage_table(outcome, limit=None):
    """一组实验的测量明细。"""
    recs = outcome.ledger.records[:limit] if limit else outcome.ledger.records
    rows = [[f"{m.a} ⇄ {m.b}", f"{m.skew_ns:.3f}", f"{m.sigma_ns:.3f}",
             f"{m.quality:.3f}", m.attempts, "合格" if m.ok else m.reason[:38]]
            for m in recs]
    tail = "" if limit is None or len(recs) == len(outcome.ledger.records) \
        else f"（仅显示前 {len(recs)} / {len(outcome.ledger.records)} 条）"
    return Markdown(md_table(["new ⇄ ref", "skew ns", "σ ns", "quality", "尝试", "结果"], rows)
                    + tail)


print("QXtrlTimingCali", __version__, "· 仓库", REPO, "· 产物目录", WORK)

QXtrlTimingCali 0.4.0 · 仓库 C:\Users\xl\Documents\XGM\Codes\QXtrlTimingCali · 产物目录 C:\Users\xl\Documents\XGM\Codes\QXtrlTimingCali\runs\demo


## 1 · 生成芯片配置文件，并可视化结构

只给**构型参数**（拓扑 / 形状 / 每线腔数），`pos`、`couplers`、`connections` 与全部线名自动推导。
`validate_chip` 会守住结构与引用的双向一致——之后若手工精修 JSON 漏改，这里会直接报错。

In [2]:
chip = generate_chip(num_qubits=8, topology="grid", shape=(2, 4), cavities_per_line=2,
                     tunable=True, seed=11, chip_id="chip-demo8")
save_chip(chip, WORK / "demo-chip.json")
check("芯片", *validate_chip(chip))

names = line_names(chip)
print(f"比特 {len(chip['qubits'])} · 腔 {len(chip['cavities'])} · 耦合器 {len(chip['couplers'])}"
      f" · 读取线 {len(chip['readout_lines'])} · 信号线 {len(names)} 条")
print("线名推导（比特 xy/z → 耦合器 z → 读取线 ro/ri）:", names[:4], "…", names[-4:])
show(render_chip_html(chip), height=620)

[芯片] errors=0  warnings=0
比特 8 · 腔 8 · 耦合器 10 · 读取线 4 · 信号线 34 条
线名推导（比特 xy/z → 耦合器 z → 读取线 ro/ri）: ['Q0.xy', 'Q0.z', 'Q1.xy', 'Q1.z'] … ['RL2.ro', 'RL2.ri', 'RL3.ro', 'RL3.ri']


## 2 · 生成初始随机 delay 的**真值文件**（模拟环境）

`kind="truth"` 的 `delay` 语义是**各线初始时刻的位置 L**（线缆长度 + 走线差异），这里按需求在
**0–1000 ns** 内随机。它是模拟器的隐藏真值：校准流程**不读也不改**它，只用来装载设备与事后对账。

> 注意：`L` 的极差就是待校准的失配量；范围拉大后，第 4 步的阈值（`skew_max_ns` / `d_max_ns`）
> 必须随之放宽，否则守卫会把正常的大位移判成异常。

In [3]:
LO, HI = 0.0, 1000.0
# seed 固定：否则每次 Run All 都是另一份隐藏真值，demo 数字不可复现
truth = generate_wiring(chip, kind="truth", delay_lo=LO, delay_hi=HI, seed=9)
save_wiring(truth, WORK / "demo-truth.json")
check("truth", *validate_wiring(truth, chip))

L = {e["line"]: float(e["delay"]) for e in truth["lines"]}
print(f"隐藏真值 L：min={min(L.values()):.1f}  max={max(L.values()):.1f} ns"
      f"  ·  初始失配 = {max(L.values()) - min(L.values()):.1f} ns（这就是要校掉的东西）")
display(shift_strip("模拟环境起始状态：q = L（待测 d 尚未开始累积）",
                    [(ln, L[ln], 0.0) for ln in names]))
show(render_chip_html(chip, truth), height=860)   # 节点上标出各线 L，下方附完整连线表

[truth] errors=0  warnings=0
隐藏真值 L：min=6.4  max=923.8 ns  ·  初始失配 = 917.4 ns（这就是要校掉的东西）


## 3 · 生成**待测文件**（校准的输入与输出载体）

`kind="pending"` 从全零起步，校准过程中被逐线填入累计右移 `d`。结构与 truth 完全一致
（`line` + `delay`），所以两者可逐线相加得到 `q`。

In [4]:
pending = generate_wiring(chip, kind="pending")
save_wiring(pending, WORK / "demo-pending.json")
check("pending", *validate_wiring(pending, chip))
assert all(e["delay"] == 0.0 for e in pending["lines"]), "待测文件应从全零起步"
print(f"待测文件 {len(pending['lines'])} 条线 · meta.kind={pending['meta']['kind']} · delay 全为 0")

[pending] errors=0  warnings=0
待测文件 34 条线 · meta.kind=pending · delay 全为 0


## 4 · 启动审查中心

三件套单向依赖：**执行方适配器**（谁来测）+ **会话**（当前 d 与同步块状态，读自待测文件）
+ **审查中心**（持有阈值与策略链，负责申请/取结果/裁决/回写）。

模拟设备 `SimChip` 用真值 `L` 装载；`DirectExecutor` 是同进程适配器，将来接 QXtrl 时
只需换成它的适配器实现，其余一概不动。

In [5]:
NOISE, SHOTS, REPEATS = 2.0, 200, 3
device = SimChip(L, noise_ns=NOISE, seed=5)
executor = DirectExecutor(SimExecutor(device, seed=5))
session = CalibrationSession.from_pending(WORK / "demo-pending.json")

th = Thresholds(tol_ns=0.5,           # 目标：残差 max(q)−min(q) ≤ 0.5 ns 即视为已对齐
                sigma_max_ns=1.0,     # 主判据不确定度上限
                quality_min=0.6,      # 评分下限
                skew_max_ns=1500.0,   # 物理合理范围：须 > L 的最大极差（这里上限 1000 ns）
                spread_max_ns=2.0,    # 多次重复的自洽性
                d_max_ns=3000.0,      # 逐线累计右移预算：多次落盘叠加（步骤 1-2 + 步骤 5 + 步骤 6）
                max_attempts=3)       # 同一条边最多重试次数
center = ReviewCenter(session, executor, thresholds=th,
                      pending_path=WORK / "demo-pending.json",
                      params={"n_shots": SHOTS, "repeats": REPEATS, "noise_ns": NOISE})
print(f"审查中心就绪：会话 {len(session.lines())} 条线 · 待测指纹 {session.config_hash}"
      f" · 设备位置 q=L（共 {len(device.lines())} 条）")

# 注意 d_max：分组之后会有多次落盘（步骤 1-2 含 headroom 即时写回 + 步骤 5 整批写回
# + 步骤 6 收回写回），同一条线的 d 是各次净位移之和，预算必须按“总位移上限”给，
# 否则守卫会把正常的大位移判成异常。


审查中心就绪：会话 34 条线 · 待测指纹 170f78984b13635c · 设备位置 q=L（共 34 条）


## 5 · 四组实验计划：谁先谁后、谁依赖谁

`plan_stages()` 按 **读取线 → 单比特 XY-Z → 比特间/耦合器 → 读取线最终对齐** 的顺序给出四组计划，
每组各自挑“代价最小生成树”（组内连通所需的最少边），并把依赖关系写在 `depends_on` 里：

- **组 1 `readout`**：节点只有各读取线的 `ro`/`ri`，组内只有“同线 `ri ↔ ro`”这种边，
  每条读取线是一个独立分量，按分量各自定标后就地写回；
- **组 2 `qubit`**：节点是各比特的 `xy`/`z`，边只有同比特内部的 `z ↔ xy`，
  不再把比特挂到读取线（这一步放到最后的 `readout_final`）；
- **组 3 `coupler`**：节点是比特 `xy`/`z` 与耦合器 `z`，边是相邻比特的 `Qa.z ↔ Qb.z` 与
  `CP.z ↔ Q.z`（可调）或 `CP.z ↔ Q.xy`（不可调），把整张图连起来；
- **组 4 `readout_final`**：节点是读取线的 `ro`/`ri` 与每条读取线首个腔的 `qubit.xy`，
  把读取线最终挂到已对齐的 qubit 上。

下面开始按组驱动：每格只跑一组，**跑完就停**，等下一个格子（= 上层调度器）再决定下一步——这正是 `stage_status()` / `run_group()` / `run_final()` 的用法。

每组还带一份**并行策略**（波划分）：同一条线不会出现在两条同时测的边里，所以一组边能拆成
几波、每波几条，是纯结构推导，与真机无关。真机上 LabExecutor 收到一波后可以真的并发提交，
也可以因 `hardware_hold` 独占锁而按顺序跑完——策略只是提交提示，结果按 `request_id` 归位，
对校准逻辑没有影响。

In [6]:
HEADROOM = 100.0  # 动画 demo 取 0；这里给 100 ns 用于演示“授予 → 记账 → 按需收回”那套机制
                   # （本例读取线到 qubit 世界最多差约 560 ns，步 6 走右拉分支，见步骤 6 说明）
pipe = StagedCalibrationPipeline(chip, center, readout_headroom_ns=HEADROOM)
plans = {p.name: p for p in pipe.stage_plans()}
total = sum(len(p.pairs) for p in plans.values())
print(f"{len(pipe.lines())} 条线 · 四组实验共需 {total} 次测量"
      f"（顺序：{' → '.join(plans)}）")

# 调度器视角：只看不跑的体检报告——现在能跑哪组、还差什么（跨进程也可用）
status = pipe.stage_status()
for name, g in status['groups'].items():
    print(f"[{name}] state={g['state']:<8} ready={g['ready']!s:<5} "
          f"blocked_by={g['blocked_by'] or '—'}")
print("下一步可做：", " → ".join(status["todo"]))
print()
for p in pipe.stage_plans():
    print(f"\n[{p.name}] {p.title}")
    print(f"    数据处置 {p.writeback:<9} · 依赖 {p.depends_on or '—'} · "
          f"节点 {len(p.nodes)} 条 · 最少测量 {len(p.pairs)} 条 · 连通分量 {p.expected_components}")
    print(f"    {p.why}")
    if p.parallel:
        print(f"    并行策略：{p.wave_summary}（同波内的边不共享线名，可一起提交）")
    for e in p.pairs[:6]:
        print(f"       {e.note:<26} {e.a} ⇄ {e.b}   cost={e.cost}")
    if len(p.pairs) > 6:
        print(f"       … 共 {len(p.pairs)} 条")


34 条线 · 四组实验共需 33 次测量（顺序：readout → qubit → coupler → readout_final）
[readout] state=pending  ready=True  blocked_by=—
[qubit] state=pending  ready=False blocked_by=['readout']
[coupler] state=pending  ready=False blocked_by=['qubit']
[readout_final] state=pending  ready=False blocked_by=['coupler', 'final（步 5 整批写回未生效）']
下一步可做： readout


[readout] 读取线校准（ro ↔ ri 对齐）
    数据处置 immediate · 依赖 — · 节点 8 条 · 最少测量 4 条 · 连通分量 4
    后续一切实验的读出前提：测完立刻写回，后面的实验才读得出信号
    并行策略：1 波 · 4 条边（4）（同波内的边不共享线名，可一起提交）
       RL0 读取对 ri-ro              RL0.ri ⇄ RL0.ro   cost=1.0
       RL1 读取对 ri-ro              RL1.ri ⇄ RL1.ro   cost=1.0
       RL2 读取对 ri-ro              RL2.ri ⇄ RL2.ro   cost=1.0
       RL3 读取对 ri-ro              RL3.ri ⇄ RL3.ro   cost=1.0

[qubit] 单比特 XY-Z 校准（xy ↔ z）
    数据处置 deferred  · 依赖 ('readout',) · 节点 16 条 · 最少测量 8 条 · 连通分量 8
    依赖已对齐的读取线：先记录，末尾与耦合器组一起解
    并行策略：1 波 · 8 条边（8）（同波内的边不共享线名，可一起提交）
       Q0 xy-z 对齐                 Q0.z ⇄ Q0.xy   cost=1.0
       Q1 xy-z 对齐                 

## 6 · 步骤 1-2：读取线 `ro ↔ ri` 对齐 + 授予 `headroom`——**立刻写回**

这一组是后面一切实验的前提：读取脉冲与采样窗口没对齐，X_π 做出来也读不出信号。
所以它**不能攒到最后**：测完就地按分量求解、就地落盘，并同步推进会话指纹与设备状态。

写回之后，流水线还给每条读取线**授予并右移** `headroom`（本例 100 ns，由
`StagedCalibrationPipeline(readout_headroom_ns=...)` 传入）：给后续 qubit/coupler 的右移
预留空间。这笔临时余量单独记账在会话的授予台账里，不属于校准位移，步骤 6 会按需收回。

写回之后再看磁盘文件，应当已经能读到非零的 `delay`——后续两组就站在这个新基线上。


In [7]:
before = disk_delays(WORK / "demo-pending.json")
ro = pipe.run_group("readout")

print(f"[{ro.plan.name}] {ro.plan.title} · 数据处置 {ro.plan.writeback}")
print(f"实测 {len(ro.ledger.records)} 次 · 合格 {len(ro.ledger.accepted())} "
      f"· 失败 {len(ro.ledger.failed())} · aborted={ro.aborted}")
print("求解：", ro.solve.reason)
print("回执：", ro.receipt and {k: ro.receipt[k] for k in ("written", "request_id")})

after = disk_delays(WORK / "demo-pending.json")
moved = {k: round(v - before[k], 3) for k, v in after.items() if v > before[k] + 1e-9}
print(f"\n✓ 立刻落盘：{len(moved)} 条读取线已右移（校准位移 + headroom {HEADROOM:g} ns）→ {moved}")
print(f"  会话指纹随落盘推进：{session.config_hash[:12]}… · 设备同步位移合计 "
      f"{sum(device.applied_shift(ln) for ln in device.lines()):.1f} ns")
assert not ro.aborted and ro.written and moved, "步骤 1 必须即时写回，否则后续实验没有读取基准"
display(stage_table(ro))

[readout] 读取线校准（ro ↔ ri 对齐） · 数据处置 immediate
实测 4 次 · 合格 4 · 失败 0 · aborted=None
求解： 8 条线 / 4 条观测 · 4 个分量各自定标 · 目标=各分量最右线（例 RL0.ri） · 最大残差 0.000 ns · 加 readout headroom 100.0 ns
回执： {'written': True, 'request_id': 'stage-readout-solve'}

✓ 立刻落盘：8 条读取线已右移（校准位移 + headroom 100 ns）→ {'RL0.ro': 151.5, 'RL0.ri': 100.0, 'RL1.ro': 100.0, 'RL1.ri': 434.193, 'RL2.ro': 100.0, 'RL2.ri': 156.797, 'RL3.ro': 628.215, 'RL3.ri': 100.0}
  会话指纹随落盘推进：6d1dc3b993fa… · 设备同步位移合计 1770.7 ns


| new ⇄ ref | skew ns | σ ns | quality | 尝试 | 结果 |
|---|---|---|---|---|---|
| RL0.ri ⇄ RL0.ro | 51.500 | 0.026 | 0.999 | 1 | 合格 |
| RL1.ri ⇄ RL1.ro | -334.193 | 0.111 | 0.988 | 1 | 合格 |
| RL2.ri ⇄ RL2.ro | -56.797 | 0.094 | 0.991 | 1 | 合格 |
| RL3.ri ⇄ RL3.ro | 528.215 | 0.030 | 0.999 | 1 | 合格 |

## 7 · 步骤 3：单比特 XY-Z 校准——只记录，**不写回**

读取线对齐之后，才有条件做单比特实验：把每个比特的 `xy` 与 `z` 对准。
注意这一步不再把比特挂到读取线——读取线会在**步骤 6** 再统一拉回对齐的 `qubit.xy`。

这一步的数据**先攒着**：现在写回会让步骤 4 的实验站在另一个基线上，两组 `skew` 就不再可比。
所以这里跑完之后，磁盘上的文件应当与步骤 1-2 结束时**逐字节一致**。

In [8]:
snapshot = disk_delays(WORK / "demo-pending.json")
qb = pipe.run_group("qubit")

print(f"[{qb.plan.name}] {qb.plan.title} · 数据处置 {qb.plan.writeback} · 依赖 {qb.plan.depends_on}")
print(f"实测 {len(qb.ledger.records)} 次 · 合格 {len(qb.ledger.accepted())} "
      f"· 失败 {len(qb.ledger.failed())} · aborted={qb.aborted}")

unchanged = disk_delays(WORK / "demo-pending.json") == snapshot
print("磁盘与步骤 1-2 结束时完全一致（本步骤只记录，未写回）：", unchanged)
assert unchanged and qb.receipt is None and not qb.written
display(stage_table(qb, limit=8))

[qubit] 单比特 XY-Z 校准（xy ↔ z） · 数据处置 deferred · 依赖 ('readout',)
实测 8 次 · 合格 8 · 失败 0 · aborted=None
磁盘与步骤 1-2 结束时完全一致（本步骤只记录，未写回）： True


| new ⇄ ref | skew ns | σ ns | quality | 尝试 | 结果 |
|---|---|---|---|---|---|
| Q0.z ⇄ Q0.xy | -89.665 | 0.164 | 0.974 | 1 | 合格 |
| Q1.z ⇄ Q1.xy | 728.127 | 0.017 | 1.000 | 1 | 合格 |
| Q2.z ⇄ Q2.xy | 496.338 | 0.071 | 0.995 | 1 | 合格 |
| Q3.z ⇄ Q3.xy | -817.532 | 0.146 | 0.979 | 1 | 合格 |
| Q4.z ⇄ Q4.xy | 62.289 | 0.020 | 1.000 | 1 | 合格 |
| Q5.z ⇄ Q5.xy | 338.028 | 0.011 | 1.000 | 1 | 合格 |
| Q6.z ⇄ Q6.xy | -251.467 | 0.104 | 0.989 | 1 | 合格 |
| Q7.z ⇄ Q7.xy | -567.924 | 0.122 | 0.985 | 1 | 合格 |

## 8 · 步骤 4：比特间 Z-Z / 耦合器 Z 校准——同样只记录

单比特门能跑通之后才能做双比特与耦合器时序：相邻可调用 `Q.z` 的比特做 `Qa.z ↔ Qb.z`，
耦合器 `CP.z` 挂到相邻比特（可调时挂 `Q.z`，不可调时退化为 `Q.xy`）。

它与步骤 3 之间没有写回，因此这两步观测**同基线**，可以合起来做一次全局求解。

In [9]:
snapshot = disk_delays(WORK / "demo-pending.json")
cp = pipe.run_group("coupler")

print(f"[{cp.plan.name}] {cp.plan.title} · 数据处置 {cp.plan.writeback} · 依赖 {cp.plan.depends_on}")
print(f"实测 {len(cp.ledger.records)} 次 · 合格 {len(cp.ledger.accepted())} "
      f"· 失败 {len(cp.ledger.failed())} · aborted={cp.aborted}")

unchanged = disk_delays(WORK / "demo-pending.json") == snapshot
print("磁盘与步骤 1-2 结束时完全一致（本步骤只记录，未写回）：", unchanged)
assert unchanged and cp.receipt is None and not cp.written
display(stage_table(cp, limit=8))

[coupler] 比特间 Z-Z / 耦合器 Z 校准 · 数据处置 deferred · 依赖 ('qubit',)
实测 17 次 · 合格 17 · 失败 0 · aborted=None
磁盘与步骤 1-2 结束时完全一致（本步骤只记录，未写回）： True


| new ⇄ ref | skew ns | σ ns | quality | 尝试 | 结果 |
|---|---|---|---|---|---|
| Q0.z ⇄ Q1.z | -493.385 | 0.073 | 0.995 | 1 | 合格 |
| Q0.z ⇄ Q4.z | -243.228 | 0.106 | 0.989 | 1 | 合格 |
| Q1.z ⇄ Q2.z | 363.785 | 0.040 | 0.998 | 1 | 合格 |
| Q1.z ⇄ Q5.z | 487.595 | 0.091 | 0.992 | 1 | 合格 |
| Q2.z ⇄ Q3.z | 421.957 | 0.048 | 0.998 | 1 | 合格 |
| Q2.z ⇄ Q6.z | 50.751 | 0.123 | 0.985 | 1 | 合格 |
| Q3.z ⇄ Q7.z | -76.351 | 0.022 | 1.000 | 1 | 合格 |
| CP0001.z ⇄ Q0.z | -135.326 | 0.097 | 0.991 | 1 | 合格 |（仅显示前 8 / 17 条）

## 9 · 步骤 5：把步骤 3 + 步骤 4 一起求解，整批一次写回（qubit/coupler 全局对齐）

现在才做全局优化，输入是：

1. 步骤 3、步骤 4 的全部合格观测（**同基线**，这是它们能合并求解的前提）；
2. 加权最小二乘按 1/σ² 分摊误差，目标取 qubit/coupler 中当前最右的那条线
   ⇒ `δ_i = max(o) − o_i ≥ 0`；
3. readout 线不参与这次求解，保持步骤 1-2（含 headroom）的位置不变。

这次求解只动 qubit/coupler，把所有 `xy`/`z` 对齐到一个共同时间轴。


In [10]:
# 步骤 5：步骤 3 + 步骤 4 的观测合并做一次全局最小二乘、整批一次写回
report = pipe.run_final(L)
print("求解：", report["solve"]["reason"])
print(f"锚线 {report['anchor']} · 目标位置 {report['target_ns']:.1f} ns "
      f"· 本批总位移 {report['plan_ns']:.1f} ns")
print(f"converged={report['converged']} · written={report['written']} · 中止={report['aborted']}")
print("台账：", report["ledger"])

display(Markdown(md_table(
    ["组", "实验", "数据处置", "依赖", "测量", "合格", "即时落盘"],
    [[s["name"], s["title"], s["writeback"], "、".join(s["depends_on"]) or "—",
      s["ledger"]["records"], s["ledger"]["accepted"], "是" if s["written"] else "—"]
     for s in report["stages"]])))


求解： 26 条线 / 25 条观测 · 目标=最右线 CP0405.z · 最大残差 0.000 ns
锚线 CP0405.z · 目标位置 476.4 ns · 本批总位移 12387.2 ns
converged=True · written=True · 中止=None
台账： {'records': 25, 'accepted': 25, 'failed': 0, 'requests': 0, 'aborted': None}


| 组 | 实验 | 数据处置 | 依赖 | 测量 | 合格 | 即时落盘 |
|---|---|---|---|---|---|---|
| qubit | 单比特 XY-Z 校准（xy ↔ z） | deferred | readout | 8 | 8 | — |
| coupler | 比特间 Z-Z / 耦合器 Z 校准 | deferred | qubit | 17 | 17 | — |

## 10 · 步骤 6：读取线收回 `headroom`、拉回对齐——**立刻写回**

qubit/coupler 全局对齐后，测每条读取线与其首个腔 `qubit.xy` 的 skew——这是全流程里唯一把
"读取线世界"和"qubit/coupler 世界"连起来的桥。据此做收尾求解，有两条分支（由"读取线是否被 headroom 推到最右"决定）：

- **收回分支**：headroom 把读取线推到了 qubit/coupler 世界右侧 → 读取线把授予的余量
  **往左收回**（净位移为负，走单独的 `headroom_return` 通道），对齐到 qubit.xy；
- **右拉分支**：读取线本来就在 qubit 世界左侧（本例：`L` 极差 917 ns 而 headroom 只有
  100 ns，读取线到 qubit 世界最多差约 560 ns）→ 照旧只向右拉，`headroom_return` 为空、
  授予余额原样留在台账里。

单调守卫 δ≥0 管的是**校准位移**；收回量受会话授予台账约束——不得超过步骤 2 授予的额度，
所以 `d` 永远不会低于加 headroom 之前的校准基线。想在本 demo 里看到收回分支，把
`HEADROOM` 调到 ≥ 700 ns 即可（`tests/test_stages.py::test_headroom_is_pulled_back_by_final_align`
守住这条路径）。


In [11]:
fin = pipe.run_group("readout_final")
print(f"[{fin.plan.name}] {fin.plan.title} · aborted={fin.aborted}")
print("求解：", fin.solve.reason)
print("右移 shifts：", {k: round(v, 3) for k, v in fin.solve.shifts.items()})
print("收回 headroom_return：", {k: round(v, 3) for k, v in fin.solve.headroom_return.items()})
print("会话授予台账余额：", {k: round(v, 1) for k, v in center.session.headroom.items()})
branch = ("收回 headroom（读取线被推到最右，往左拉回对齐）" if fin.solve.headroom_return
          else "只向右拉（读取线在 qubit 世界左侧，headroom 留在台账里下次生效）")
print("收尾分支：", branch)
assert not fin.aborted and fin.written, "步骤 6 必须把读取线拉回对齐并落盘"

# 与 pipe.run() 的收尾一致：把组 4 追加进报表，并在“所有线都已参与”后重算残差
report["stages"].append(fin.to_dict())
report["written"] = bool(report["written"] or fin.written)
report["isolated"] = []
report["residual"] = position_residual(L, center.session.delays)
report["converged"] = bool(report["solve"]["offsets"]) and not report["aborted"]
report["aligned"] = bool(report["converged"]
                         and report["residual"]["residual_ns"] <= th.tol_ns
                         and not report["aborted"])
print(f"\n✓ 读取线已拉回对齐：残差 {report['residual']['residual_ns']:.3f} ns · "
      f"aligned={report['aligned']}")

(WORK / "pipeline-report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
center.session.save(WORK / "session.json")
print("产物：", sorted(p.name for p in WORK.iterdir()))


[readout_final] 读取线最终对齐（收回 headroom） · aborted=None
求解： 30 条线 / 29 条观测 · 目标=最右线 CP0001.z · 最大残差 0.000 ns · 读取线 ro/ri 同步增减（4 条 ri 随动）
右移 shifts： {'Q0.xy': 0.0, 'Q2.xy': 0.0, 'Q4.xy': 0.0, 'Q6.xy': 0.0, 'RL0.ro': 97.845, 'RL1.ro': 401.534, 'RL2.ro': 556.986, 'RL3.ro': 14.032, 'Q0.z': 0.0, 'Q1.xy': 0.0, 'Q1.z': 0.0, 'Q2.z': 0.0, 'Q3.xy': 0.0, 'Q3.z': 0.0, 'Q4.z': 0.0, 'Q5.xy': 0.0, 'Q5.z': 0.0, 'Q6.z': 0.0, 'Q7.xy': 0.0, 'Q7.z': 0.0, 'CP0001.z': 0.0, 'CP0004.z': 0.0, 'CP0102.z': 0.0, 'CP0105.z': 0.0, 'CP0203.z': 0.0, 'CP0206.z': 0.0, 'CP0307.z': 0.0, 'CP0405.z': 0.0, 'CP0506.z': 0.0, 'CP0607.z': 0.0, 'RL0.ri': 97.845, 'RL1.ri': 401.534, 'RL2.ri': 556.986, 'RL3.ri': 14.032}
收回 headroom_return： {}
会话授予台账余额： {'RL0.ri': 100.0, 'RL0.ro': 100.0, 'RL1.ri': 100.0, 'RL1.ro': 100.0, 'RL2.ri': 100.0, 'RL2.ro': 100.0, 'RL3.ri': 100.0, 'RL3.ro': 100.0}
收尾分支： 只向右拉（读取线在 qubit 世界左侧，headroom 留在台账里下次生效）

✓ 读取线已拉回对齐：残差 0.466 ns · aligned=True
产物： ['demo-chip.json', 'demo-pending-abort.json', 'demo-pending-a

## 11 · 与保存的真值对账，确认校准效果

**回读磁盘上的待测文件**（不是内存里的会话），与 truth 文件逐线相加得 `q = L + d`，
残差取 `max(q) − min(q)`。

只允许右移 ⇒ 结果不是"把所有 `d` 校成 0"，而是把所有线推到同一条时间轴上（整体位置可以偏右）；
所以评价标准是**一致性**（残差），不是 `d` 的大小。六步流程把读取线的最终对齐放到 qubit/coupler
全局对齐之后：步骤 6 依据实测 skew 把读取线拉回（收回步骤 2 的 headroom），
避免读取线被 qubit 门“撞到”。

残差的量级由测量精度决定：`σ` 越小（多 shot / 多重复）残差越小，而它**不会**再像旧流程那样
随台阶数增长而累积。


In [12]:
saved_L = {e["line"]: float(e["delay"])
           for e in json.loads((WORK / "demo-truth.json").read_text(encoding="utf-8"))["lines"]}
final = json.loads((WORK / "demo-pending.json").read_text(encoding="utf-8"))
saved_d = {e["line"]: float(e["delay"]) for e in final["lines"]}
check("校准后的待测文件", *validate_wiring(final, chip))

res = position_residual(saved_L, saved_d)
misfit = max(saved_L.values()) - min(saved_L.values())
print(f"初始失配 {misfit:.1f} ns  →  校准后残差 max(q)−min(q) = {res['residual_ns']:.3f} ns"
      f"（单步目标 tol={th.tol_ns} ns）")
print(f"q 落在 [{res['min_ns']:.1f}, {res['max_ns']:.1f}] ns · 最大累计右移 max(d)"
      f" = {max(saved_d.values()):.1f} ns ≤ d_max={th.d_max_ns:g} ns")
print(f"全部线都参与求解：{not report['isolated']} · 隔离区：{report['isolated']}")
assert res["residual_ns"] * 100 < misfit, "残差应比初始失配小两个数量级"
assert report["converged"] and not report["aborted"]

display(shift_strip(f"校准前：q = L（失配 {misfit:.1f} ns）", [(n, saved_L[n], 0.0) for n in names]))
display(shift_strip(f"校准后：q = L + d（残差 {res['residual_ns']:.3f} ns）",
                    [(n, saved_L[n], saved_d[n]) for n in names]))
show(render_chip_html(chip, final), height=860)

[校准后的待测文件] errors=0  warnings=0
初始失配 917.4 ns  →  校准后残差 max(q)−min(q) = 0.466 ns（单步目标 tol=0.5 ns）
q 落在 [923.3, 923.8] ns · 最大累计右移 max(d) = 917.0 ns ≤ d_max=3000 ns
全部线都参与求解：True · 隔离区：[]


## 附 · 步骤 1 就失败时会怎样

把执行方的自报评分压到 `0.1`（低于 `quality_min`），读取线这一步重试到用尽 `max_attempts`
后全部判废。此时**后面的步骤根本不会开始**——没有读取基准，单比特与耦合器实验无从谈起，
这正是“分组 + 依赖”的意义：编排器按 `depends_on` 停在第一阶段，待测文件一个字节都不会被改。

（换成因某条线 `d` 超预算而在末尾被守卫拒绝，四道守卫同样会把**整批**一起拒掉，
绝不会写成半成品。）

In [13]:
bad_pending = WORK / "demo-pending-abort.json"
save_wiring(generate_wiring(chip, kind="pending"), bad_pending)
bad_center = ReviewCenter(
    CalibrationSession.from_pending(bad_pending),
    DirectExecutor(SimExecutor(SimChip(L, noise_ns=NOISE, seed=9), seed=9,
                               force={"quality": 0.1})),
    thresholds=th, pending_path=bad_pending,
    params={"n_shots": SHOTS, "repeats": REPEATS, "noise_ns": NOISE})
bad_report = StagedCalibrationPipeline(chip, bad_center).run(L)

untouched = all(e["delay"] == 0.0
                for e in json.loads(bad_pending.read_text(encoding="utf-8"))["lines"])
print("中止原因：", bad_report["aborted"])
print("实际执行的阶段：", [s["name"] for s in bad_report["stages"]],
      "（步骤 3-6 因缺少读取基准而未执行）")
print(f"失败边 {len(bad_report['failed'])} 条 · 隔离区 {len(bad_report['isolated'])} 条 "
      f"· written={bad_report['written']}")
print("待测文件保持原样（全零）：", untouched)
assert bad_report["aborted"] and not bad_report["written"] and untouched

中止原因： 读取线校准（ro ↔ ri 对齐） 无合格观测
实际执行的阶段： ['readout'] （步骤 3-6 因缺少读取基准而未执行）
失败边 4 条 · 隔离区 0 条 · written=False
待测文件保持原样（全零）： True


## 小结与下一步

- 本 demo 覆盖了 v0.4 的六步主线：**生成 → 校验 → 可视化 → 四组实验计划 →
  读取线对齐并加 headroom → 单比特 XY-Z / 耦合器 Z-Z 只记录 → qubit/coupler 全局求解写回 →
  读取线收尾对齐（右拉 / 按需收回 headroom）→ 真值对账**；
- 四组顺序是物理依赖：`ro/ri` 对齐 → headroom → 单比特 `XY-Z` → 比特间/耦合器 `Z-Z` → 收回 headroom 拉回对齐；
  前一组没就绪，编排器不会让后一组开始；
- `headroom` 可按经验调整（本 notebook 与动画 demo 一致取 `100` ns，典型可设 `1000` ns）：
  它由流水线自己授予、步骤 6 按需收回，不需要为此调整 `LO, HI` 或读取线的 L；
- 想看"多测几条冗余边能不能压残差"：把某组的 `pairs` 换成"生成树 + 若干额外边"再喂给
  同一个 `run_group()`，此时 `solve` 的最大残差会变成发现坏数据的探针；
- **分阶段驱动**：本文一格跑一组，正是在演示上层调度器的用法——真实流程里读完 `ro/ri`
  还要插 S21 / spectrum / scatter / power Rabi，之后随时可以停下来，下次用
  `stage_status()` 看进度、`run_group()` / `run_final()` 接着推；跨进程也成立；
- 接真实执行端（QXtrl）时的三件事：实现它的 `ExecutorAdapter`、把线名映射到板卡通道留在执行侧、
  **延时字段的唯一写入者仍是审查中心**；
- 架构全景与接入边界见 `docs/architecture-v0.3.2.html`（`python docs/gen_architecture.py` 重生成）。
